# ray.wait() 是 Ray 框架中用于异步控制的核心函数，它是实现高性能分布式计算的关键。
- <font color='red'>一句话建议：当你发现程序在“傻等”某个慢任务时，就是引入 ray.wait() 的最佳时机</font>

- 如果说 ray.get() 是“死板地等待所有任务结束”，那么 ray.wait() 就是“灵活地处理最先完成的任务”。它允许你在任务执行过程中动态响应，而不是被动等待。
## 🎯 核心作用：为什么需要 ray.wait()？

1. 解决 ray.get() 的阻塞问题
当你有一批任务（例如 10 个），使用 ray.get(list_of_refs) 时，你必须等到最慢的那个任务（Straggler）完成，才能拿到所有结果。
    - 问题：如果前 9 个任务 1 秒就完成了，第 10 个任务卡住了要跑 1 小时，你的程序就会白白干等 1 小时。
    - <font color='red'>ray.wait() 的解法：它会告诉你“嘿，前 9 个已经好了，你先拿去处理，别管那个慢的”。</font>

2. 实现动态负载均衡

<font color='red'>在 ray.get 模式下，任务分配是静态的。而在 ray.wait 模式下，你可以实现“谁先做完，就给谁派新活”的模式，这对于任务执行时间不确定的场景（如机器学习推理、网页爬虫）至关重要。</font>

## ⚙️ 函数签名与参数详解

In [ ]:
ray.wait(object_refs, num_returns=1, timeout=None, fetch_local=True)

- object_refs (列表):
    - 你正在监听的一组 ObjectRef（即未完成的远程任务列表）。
- <font color='red'>num_returns (整数，默认=1):</font>
    - 核心参数。告诉 ray.wait 你希望至少等到多少个任务完成才返回。
    - <font color='red'>例如：设置为 1，只要有一个任务做完就返回；设置为 5，则必须凑齐 5 个做完的任务才返回。</font>
- timeout (浮点数，秒，默认=None):
    - 安全参数。最多等待多少秒。
    - 强烈建议设置：如果任务卡死（死循环或死锁），不设置超时会导致程序永远挂起。如果超时，它会返回当前已经完成的任务（可能少于 num_returns）。
- fetch_local (布尔值，默认=True):
    - <font color='red'>是否将数据拉取到本地。通常保持默认即可。</font>

## 🔄 返回值：双列表机制

ray.wait() 总是返回两个列表：

- ready_ids (已完成列表): 包含已经执行完毕的任务的 ObjectRef。你可以对这些引用调用 ray.get() 获取真实数据。
- remaining_ids (未完成列表): 包含还在运行或尚未开始的任务的 ObjectRef。你可以把它们放回下一次的 ray.wait 中继续监听。

## 🚀 典型应用场景与代码模式

### 场景一：流式处理 (Stream Processing)
“来一个处理一个，不等所有人”。适用于任务执行时间差异巨大的情况。

In [ ]:
import ray
import time

ray.init()

@ray.remote
def slow_task(i):
    # 模拟随机执行时间
    time.sleep(i) 
    return f"任务 {i} 完成"

# 提交 5 个任务，耗时分别是 1s, 2s, 3s, 4s, 5s
futures = [slow_task.remote(i) for i in [1, 2, 3, 4, 5]]

# 只要还有未完成的任务，就循环监听
while futures:
    # 等待任意 1 个任务完成
    done, futures = ray.wait(futures, num_returns=1, timeout=10)
    
    # 立即处理完成的任务
    for result_ref in done:
        result = ray.get(result_ref)
        print(f"收到结果: {result}")
        # 在这里可以提交新任务，实现动态流水线

##### 输出顺序：
- <font color='red'>你会看到结果是按 1, 2, 3, 4, 5 的顺序出来的，而不是等 5 秒后一次性出来。</font>

### 场景二：投机执行 (Speculative Execution)
“谁慢就杀谁，换个新的跑”。这是大数据系统（如 Spark）常用的优化手段，用于处理“长尾任务”。

In [ ]:
# 假设我们有两个任务副本在跑，谁先跑完用谁
primary = slow_task.remote(5) # 主任务，预计 5 秒
backup = slow_task.remote(5)  # 备份任务，预计 5 秒

# 等待任意一个完成
done, remaining = ray.wait([primary, backup], num_returns=1)

# 获取结果
result = ray.get(done[0])
print(f"最终结果: {result}")

# 既然已经拿到了结果，剩下的那个慢任务就没用了，取消它以释放资源
ray.cancel(remaining[0]) 

### 场景三：批量聚合 (Batching)
“凑够一拨再处理”。适用于需要批量写入数据库或进行矩阵运算的场景。

In [ ]:
futures = [slow_task.remote(i) for i in range(100)]

while futures:
    # 每次凑够 10 个结果再处理，提高批量操作效率
    done, futures = ray.wait(futures, num_returns=10, timeout=60)
    
    if len(done) == 10:
        results = ray.get(done)
        print(f"批量处理 10 条数据: {results}")
    else:
        print("超时或任务结束，处理剩余数据...")

## ⚠️ 注意事项与陷阱

- 结果乱序：

<font color='red'>ray.wait 返回的结果顺序取决于任务完成的速度，而不是你提交的顺序。如果你需要保持顺序，必须在数据中携带索引（例如返回 (index, result)），然后在处理时重新排序。</font>

- 超时处理：

<font color='red'>如果设置了 timeout，返回的 done 列表长度可能小于 num_returns。代码中必须处理这种情况（如上面的批量聚合示例），否则可能导致逻辑错误。</font>

- 不要滥用：

<font color='red'>如果你的任务执行时间都很稳定，且必须等所有结果都齐了才能进行下一步（例如矩阵乘法）</font>，直接用 ray.get 更简单、代码更清晰。ray.wait 会增加代码的复杂度。

| 特性 | `ray.get()` | `ray.wait()` |
| :--- | :--- | :--- |
| 等待行为 | 阻塞，直到所有任务完成 | 非阻塞，直到指定数量任务完成 |
| 适用场景 | 简单的并行计算，结果强依赖 | 动态调度、流式处理、长尾任务优化 |
| 效率 | 受限于最慢的任务 | 充分利用计算资源，减少空转 |
| 代码复杂度 | 低 | 中高（需要循环和状态管理） |

